# Colab Model Station

An automated, reproducible deployment framework for Google Colab GPU environments.
Supports multiple inference backends (**FastAPI + Diffusers** / **ComfyUI Headless API**) with dynamic **FLUX**, **SDXL**, and **LoRA** hot-swapping over **Tailscale Mesh** or **Cloudflare Tunnels**.

---

## Step 0: Repository Setup & Hardware Inspection
Clones `colab-inference-node` to local runtime storage (`/content/colab-inference-node`), injects Colab Secrets, and displays GPU allocation.

In [ ]:
# ==============================================================================
# Step 0: Clone Repository, Inject Secrets, and Inspect GPU
# ==============================================================================
import os
from google.colab import userdata

# 1. Inject Secrets from Colab Secrets store (if configured)
try:
    os.environ['TAILSCALE_AUTHKEY'] = userdata.get('TAILSCALE_AUTHKEY')
except Exception:
    pass

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

# 2. Clone or update repository
!git clone https://github.com/yixuanw99/colab-inference-node.git /content/colab-inference-node 2>/dev/null || (cd /content/colab-inference-node && git pull)

# 3. Hardware Profile
!cd /content/colab-inference-node && bash engine.sh status


## Step 1: Environment Provisioning

Install the required packages based on your target workflow:
- **Diffusers only** (`bash setup.sh diffusers`): Installs FastAPI, Uvicorn, Diffusers, Transformers, PEFT, and Accelerate.
- **ComfyUI only** (`bash setup.sh comfyui`): Clones ComfyUI core and installs execution dependencies.
- **Full Stack** (`bash setup.sh all`): Installs Diffusers, Tailscale, Cloudflared, and system utilities.

In [ ]:
%%bash
cd /content/colab-inference-node

# Select target provisioning mode: diffusers | comfyui | all
bash setup.sh all


## Hardware, Resource Footprint & Model Matrix

Match your Colab GPU tier and VRAM budget with the recommended model precision and offload strategy:

| GPU Tier | VRAM | Engine | Model | Precision | Offload Mode | Steps | Latency (est.) |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **NVIDIA T4** | 15 GB | **Diffusers** | `black-forest-labs/FLUX.1-schnell` | FP8 | Sequential | 4 | ~35s - 50s |
| **NVIDIA T4** | 15 GB | **Diffusers** | `stabilityai/stable-diffusion-xl-base-1.0` | FP16 | Model | 30 | ~12s - 18s |
| **NVIDIA L4** | 24 GB | **Diffusers** | `black-forest-labs/FLUX.1-schnell` | FP8 | Model Offload | 4 | ~12s - 18s |
| **NVIDIA L4** | 24 GB | **Diffusers** | `black-forest-labs/FLUX.1-dev` | FP8 | Model Offload | 28 | ~45s - 65s |
| **NVIDIA A100** | 40/80 GB | **Diffusers** | `black-forest-labs/FLUX.1-schnell` | BF16 | None (GPU) | 4 | ~4s - 8s |
| **NVIDIA A100** | 40/80 GB | **Diffusers** | `black-forest-labs/FLUX.1-dev` | BF16 | None (GPU) | 28 | ~15s - 25s |

---

## Deployment Option A: Diffusers Server + Tailscale Mesh (Recommended)

### Characteristics
- **Ingress**: Encrypted Private WireGuard Mesh (`http://colab-model-station:8000`)
- **Requirement**: `TAILSCALE_AUTHKEY` in Colab Secrets.
- **Best For**: Private, headless programmatic generation directly from your local terminal or IDE.

In [ ]:
%%bash
cd /content/colab-inference-node

# 1. Connect to Tailscale Mesh in Userspace mode
bash engine.sh tunnel tailscale up "$TAILSCALE_AUTHKEY"

# 2. Launch FastAPI Diffusers inference server (FLUX.1-schnell)
bash engine.sh diffusers start --model "black-forest-labs/FLUX.1-schnell" --precision fp8

# 3. Proxy port 8000 to Tailnet
bash engine.sh tunnel tailscale serve 8000

# 4. Arm Idle Watchdog (30 minute compute unit protection)
bash engine.sh watchdog start --port 8000 --timeout 1800

echo "================================================================"
echo "Diffusers Tailnet Endpoint: http://colab-model-station:8000"
echo "================================================================"


## Deployment Option B: Diffusers Server + Cloudflare Public Tunnel

### Characteristics
- **Ingress**: Temporary Public HTTPS Endpoint (`https://*.trycloudflare.com`)
- **Requirement**: Zero configuration. No accounts required.
- **Best For**: Quick testing or sharing endpoints without Tailscale client installed on workstation.

In [ ]:
%%bash
cd /content/colab-inference-node

# 1. Launch FastAPI Diffusers server
bash engine.sh diffusers start --model "black-forest-labs/FLUX.1-schnell" --precision fp8

# 2. Start Cloudflare Tunnel on port 8000
bash engine.sh tunnel cloudflare up 8000

# 3. Arm Idle Watchdog
bash engine.sh watchdog start --port 8000 --timeout 1800


## Step 2: In-Notebook Verification & Dynamic LoRA Switching

Verify the live inference endpoint, generate a test image, and dynamically attach a LoRA adapter.

In [ ]:
import requests
import base64
from IPython.display import display, Image

BASE_URL = "http://127.0.0.1:8000"

# 1. Health & VRAM Check
health = requests.get(f"{BASE_URL}/health").json()
print(f"Service Status: {health['status']} | GPU: {health['gpu_name']} | VRAM: {health['vram_allocated_mb']}MB")

# 2. Generate Image
payload = {
    "prompt": "A futuristic scientific laboratory on Mars, hyperrealistic, cinematic lighting",
    "num_inference_steps": 4,
    "width": 1024,
    "height": 1024,
    "return_base64": True
}
res = requests.post(f"{BASE_URL}/v1/images/generations", json=payload).json()
print(f"Generated in {res['generation_time_sec']}s")

img_b64 = res['images'][0]['base64_data']
display(Image(base64.b64decode(img_b64)))


## Step 3: Compute Unit Protection & Runtime Teardown

> **CRITICAL**: Closing your browser tab does **NOT** stop your Colab GPU instance. Paid compute units will continue to drain until hard timeout.

Run this final cell to release all GPU VRAM and terminate the Colab instance immediately.

In [ ]:
# ==============================================================================
# Step 3: Terminate Colab Runtime and Stop Billing
# ==============================================================================
from google.colab import runtime
print("Disconnecting Google Colab runtime and releasing compute units...")
runtime.unassign()
